In [ ]:
import cdsapi
import os

In [ ]:
import os
import cdsapi
from dotenv import load_dotenv

load_dotenv()

URL = "https://cds.climate.copernicus.eu/api"
KEY = os.getenv("CDS_API_KEY") 

if not KEY:
    raise ValueError("Error: CDS_API_KEY not found in environment variables. Check your .env file.")

output_folder = "data_raw"
os.makedirs(output_folder, exist_ok=True)

client = cdsapi.Client(url=URL, key=KEY)


for year in range(2000, 2021):
    filename = f"nazare_{year}.nc"
    target_path = os.path.join(output_folder, filename)
    
    print(f"--- Processing year: {year} ---")

    request = {
        "product_type": ["reanalysis"],
        "variable": [
            "10m_u_component_of_wind",
            "10m_v_component_of_wind",
            "2m_dewpoint_temperature",
            "2m_temperature",
            "mean_sea_level_pressure",
            "mean_wave_direction",
            "mean_wave_period",
            "sea_surface_temperature",
            "significant_height_of_combined_wind_waves_and_swell",
            "total_precipitation",
            "instantaneous_surface_sensible_heat_flux",
            "peak_wave_period"
        ],
        "year": [str(year)], 
        "month": [f"{m:02d}" for m in range(1, 13)],
        "day": [f"{d:02d}" for d in range(1, 32)],
        "time": ["00:00", "06:00", "12:00", "18:00"],
        "data_format": "netcdf",
        "download_format": "unarchived",
        "area": [39.7, -9.8, 39.5, -9.3]
    }
    
    try:
        if os.path.exists(target_path):
            print(f"File {filename} already exists. Skipping...")
            continue
            
        print(f"Starting download: {filename}")
        client.retrieve("reanalysis-era5-single-levels", request, target_path)
        print(f"Success: {filename}")
        
    except Exception as e:
        print(f"Error downloading data for year {year}: {e}")

print("All tasks completed successfully!")